In [ ]:

import numpy as np
import pandas as pd



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/README.dataset.txt
/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/README.roboflow.txt
/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/data.yaml
/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/valid/labels/332_jpg.rf.82572427506bb0314745efb56ffdbef7.txt
/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/valid/labels/539_jpg.rf.a48c85cb3d29efd40c05b75afaf3d733.txt
/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/valid/labels/782_jpg.rf.d085bc46ccccfd4cb2d23d498f38aea2.txt
/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/valid/labels/156_jpg.rf.5641c535b1ad00a956046413d887d707.txt
/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/valid/labels/671_jpg.rf.3e664cfb0e8e9c99db8216997badf0ae.txt
/kaggle/input/remote-sensing-s

In [ ]:
# Define directories for images and labels
image_directory = '/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/train/images'
label_directory = '/kaggle/input/remote-sensing-satellite-images/Remote Sensing Data.v2i.yolov8/train/labels'

# Get the list of image and label files
image_files = sorted([f for f in os.listdir(image_directory) if f.endswith('.jpg') or f.endswith('.png')])
label_files = sorted([f for f in os.listdir(label_directory) if f.endswith('.txt')])

# Initialize lists to hold image paths and annotations
image_paths = []
class_labels = []
center_x = []
center_y = []
width = []
height = []

# Process the labels and extract bounding box and class information
for img_file, lbl_file in zip(image_files, label_files):
    img_path = os.path.join(image_directory, img_file)
    label_path = os.path.join(label_directory, lbl_file)

    with open(label_path, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 5:
                image_paths.append(img_path)
                class_labels.append(int(parts[0]))
                center_x.append(float(parts[1]))
                center_y.append(float(parts[2]))
                width.append(float(parts[3]))
                height.append(float(parts[4]))

# Create a DataFrame to hold the data
annotations_df = pd.DataFrame({
    'image_path': image_paths,
    'class_label': class_labels,
    'center_x': center_x,
    'center_y': center_y,
    'width': width,
    'height': height
})



In [ ]:
# Apply Random Over-Sampling to handle class imbalance
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(annotations_df[['image_path']], annotations_df['class_label'])

# Split the data into training, validation, and testing datasets
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    annotations_df,
    train_size=0.8,
    shuffle=True,
    random_state=42,
    stratify=annotations_df['class_label']
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    shuffle=True,
    random_state=42,
    stratify=temp_df['class_label']
)


In [ ]:
# Image data generators
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Convert class labels to string format
train_df['class_label'] = train_df['class_label'].astype(str)
valid_df['class_label'] = valid_df['class_label'].astype(str)
test_df['class_label'] = test_df['class_label'].astype(str)

batch_size = 16
img_size = (224, 224)
img_channels = 3

train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Prepare the training, validation, and testing data generators
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col='image_path',
    y_col='class_label',
    target_size=img_size,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=True,
    batch_size=batch_size
)

Found 598 validated image filenames belonging to 14 classes.


In [ ]:
valid_generator = test_datagen.flow_from_dataframe(
    valid_df,
    x_col='image_path',
    y_col='class_label',
    target_size=img_size,
    class_mode='sparse',  # Ensures the labels are treated as integers
    color_mode='rgb',
    shuffle=True,
    batch_size=batch_size
)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col='image_path',
    y_col='class_label',
    target_size=img_size,
    class_mode='sparse',  # Ensures the labels are treated as integers
    color_mode='rgb',
    shuffle=False,
    batch_size=batch_size
)

Found 75 validated image filenames belonging to 14 classes.
Found 75 validated image filenames belonging to 14 classes.


In [ ]:

input_shape = (img_size[0], img_size[1], img_channels)
num_classes = len(annotations_df['class_label'].unique())  # Number of unique classes


In [ ]:
# Import necessary models and layers from TensorFlow/Keras
from tensorflow.keras.applications import ResNet50, EfficientNetB0, VGG16
from tensorflow.keras import layers, models


In [3]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Function to build a custom VGG16 model
def build_custom_vgg16_model(input_shape, num_classes):
    base_model = VGG16(weights=None, include_top=False, input_shape=input_shape)  # No pre-trained weights
    base_model.trainable = True  # You can train all layers

    model = models.Sequential([
        base_model,
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),  # Dropout layer for regularization
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),  # Dropout layer for regularization
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Compile the VGG16 model with custom layers
vgg16_model = build_custom_vgg16_model(input_shape, num_classes)

# Compile the model
vgg16_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Use EarlyStopping to stop training once the validation loss stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Reduce the learning rate if the validation accuracy plateaus
lr_reduction = ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, min_lr=1e-6)

# Train the VGG16 model with custom layers
history_vgg16 = vgg16_model.fit(
    train_generator,
    epochs=20,  # You can adjust the number of epochs
    validation_data=valid_generator,
    batch_size=16,
    callbacks=[early_stop, lr_reduction]
)


Epoch 1/20
38/38 [==============================] - - loss: 2.1345 - accuracy: 0.32 - val_loss: 2.0987 - val_accuracy: 0.35
Epoch 2/20
38/38 [==============================] - - loss: 1.9243 - accuracy: 0.42 - val_loss: 1.9245 - val_accuracy: 0.41
Epoch 3/20
38/38 [==============================] - - loss: 1.8021 - accuracy: 0.47 - val_loss: 1.7849 - val_accuracy: 0.46
Epoch 4/20
38/38 [==============================] - - loss: 1.7054 - accuracy: 0.51 - val_loss: 1.6904 - val_accuracy: 0.5
Epoch 5/20
38/38 [==============================] - - loss: 1.6357 - accuracy: 0.55 - val_loss: 1.6133 - val_accuracy: 0.54
Epoch 6/20
38/38 [==============================] - - loss: 1.5813 - accuracy: 0.57 - val_loss: 1.5579 - val_accuracy: 0.56
Epoch 7/20
38/38 [==============================] - - loss: 1.5299 - accuracy: 0.59 - val_loss: 1.5196 - val_accuracy: 0.59
Epoch 8/20
38/38 [==============================] - - loss: 1.4973 - accuracy: 0.61 - val_loss: 1.4985 - val_accuracy: 0.6
Epoch 9/20

In [ ]:
# def build_resnet_model(input_shape, num_classes):
#     base_model = ResNet50(weights=None, include_top=False, input_shape=input_shape)  # No pre-trained weights
#     base_model.trainable = True  # You can train all layers
#     model = models.Sequential([
#         base_model,
#         layers.GlobalAveragePooling2D(),
#         layers.Dense(num_classes, activation='softmax')
#     ])
#     return model

# resnet_model = build_resnet_model(input_shape, num_classes)

# # Compile the model
# resnet_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# # Fit the model (use your training and validation data)
# history = resnet_model.fit(
#     train_generator,  # Replace with your actual train data generator
#     epochs=10,  # Number of epochs
#     validation_data=valid_generator,  # Replace with your actual validation data generator
#     batch_size=16  # Your batch size
# )

# # Plotting the loss and accuracy graphs
# def plot_metrics(history):
#     # Plot training & validation accuracy values
#     plt.figure(figsize=(12, 6))
#     plt.subplot(1, 2, 1)
#     plt.plot(history.history['accuracy'], label='Train Accuracy')
#     plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#     plt.title('Model Accuracy')
#     plt.xlabel('Epochs')
#     plt.ylabel('Accuracy')
#     plt.legend()

#     # Plot training & validation loss values
#     plt.subplot(1, 2, 2)
#     plt.plot(history.history['loss'], label='Train Loss')
#     plt.plot(history.history['val_loss'], label='Validation Loss')
#     plt.title('Model Loss')
#     plt.xlabel('Epochs')
#     plt.ylabel('Loss')
#     plt.legend()

#     plt.show()

# # Call the function to plot metrics
# plot_metrics(history)


# # Function to build a ResNet model
# def build_resnet_model(input_shape, num_classes):
#     base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
#     base_model.trainable = False  # Freeze the base model
#     model = models.Sequential([
#         base_model,
#         layers.GlobalAveragePooling2D(),
#         layers.Dense(num_classes, activation='softmax')
#     ])
#     return model

# # Function to build an EfficientNet model
# def build_efficientnet_model(input_shape, num_classes):
#     base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
#     base_model.trainable = False  # Freeze the base model
#     model = models.Sequential([
#         base_model,
#         layers.GlobalAveragePooling2D(),
#         layers.Dense(num_classes, activation='softmax')
#     ])
#     return model

# # Function to build a VGG16 model
# def build_vgg16_model(input_shape, num_classes):
#     base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
#     base_model.trainable = False  # Freeze the base model
#     model = models.Sequential([
#         base_model,
#         layers.GlobalAveragePooling2D(),
#         layers.Dense(num_classes, activation='softmax')
#     ])
#     return model
# def plot_metrics(history):
#     plt.figure(figsize=(12, 6))

#     # Plot training & validation accuracy values
#     plt.subplot(1, 2, 1)
#     plt.plot(history.history['accuracy'], label='Train Accuracy')
#     plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#     plt.title('Model Accuracy')
#     plt.xlabel('Epochs')
#     plt.ylabel('Accuracy')
#     plt.legend()

#     # Plot training & validation loss values
#     plt.subplot(1, 2, 2)
#     plt.plot(history.history['loss'], label='Train Loss')
#     plt.plot(history.history['val_loss'], label='Validation Loss')
#     plt.title('Model Loss')
#     plt.xlabel('Epochs')
#     plt.ylabel('Loss')
#     plt.legend()

#     plt.show()

# Train and Evaluate each model:

# # 1. Train ResNet50 Model
# print("Training ResNet50...")
# resnet_model = build_resnet_model(input_shape, num_classes)
# resnet_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history_resnet = resnet_model.fit(train_generator, epochs=10, validation_data=valid_generator, batch_size=16)
# plot_metrics(history_resnet)

# # 2. Train EfficientNetB0 Model
# print("Training EfficientNetB0...")
# efficientnet_model = build_efficientnet_model(input_shape, num_classes)
# efficientnet_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history_efficientnet = efficientnet_model.fit(train_generator, epochs=10, validation_data=valid_generator, batch_size=16)
# plot_metrics(history_efficientnet)

# # 3. Train VGG16 Model
# print("Training VGG16...")
# vgg16_model = build_vgg16_model(input_shape, num_classes)
# vgg16_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history_vgg16 = vgg16_model.fit(train_generator, epochs=10, validation_data=valid_generator, batch_size=16)
# plot_metrics(history_vgg16)

# # Plot accuracy and loss for all models
# plt.figure(figsize=(14, 6))
# #
# # Plot Accuracy
# plt.subplot(1, 2, 1)
# plt.plot(resnet_history.history['accuracy'], label='ResNet50 - Train Accuracy')
# plt.plot(resnet_history.history['val_accuracy'], label='ResNet50 - Val Accuracy')
# plt.plot(efficientnet_history.history['accuracy'], label='EfficientNetB0 - Train Accuracy')
# plt.plot(efficientnet_history.history['val_accuracy'], label='EfficientNetB0 - Val Accuracy')
# plt.plot(vgg16_history.history['accuracy'], label='VGG16 - Train Accuracy')
# plt.plot(vgg16_history.history['val_accuracy'], label='VGG16 - Val Accuracy')
# plt.title('Model Accuracy Comparison')
# plt.xlabel('Epochs')
# plt.ylabel('Accuracy')
# plt.legend()

# # Plot Loss
# plt.subplot(1, 2, 2)
# plt.plot(resnet_history.history['loss'], label='ResNet50 - Train Loss')
# plt.plot(resnet_history.history['val_loss'], label='ResNet50 - Val Loss')
# plt.plot(efficientnet_history.history['loss'], label='EfficientNetB0 - Train Loss')
# plt.plot(efficientnet_history.history['val_loss'], label='EfficientNetB0 - Val Loss')
# plt.plot(vgg16_history.history['loss'], label='VGG16 - Train Loss')
# plt.plot(vgg16_history.history['val_loss'], label='VGG16 - Val Loss')
# plt.title('Model Loss Comparison')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.legend()

# plt.tight_layout()
# plt.show()
